# Module 1: Let's Understand Search

Qdrant Beginners Course, follow-along notebook.

Course page: https://qdrant.tech/course/beginners/module-1/

Keyword search matches words. Semantic search matches meaning. This module covers embeddings, cosine similarity, and where similarity alone still fails.


In [ ]:
!pip install -q fastembed numpy

## Why keyword search struggles

- Different words, same meaning: "car repair" vs "automobile maintenance"
- Same word, different meaning: "Apple stock" (fruit or finance)
- Same words, different order: "dog bites man" vs "man bites dog"

## Embeddings

An embedding is a vector: a list of numbers that captures meaning. Similar meaning, similar vectors.


In [ ]:
from fastembed import TextEmbedding

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
query_vec = list(model.embed(["car repair"]))[0]
doc_vec   = list(model.embed(["automobile maintenance"]))[0]

print(len(query_vec), len(doc_vec))
print(query_vec[:5])
print(doc_vec[:5])

## Cosine similarity

Measures the angle between two vectors. Closer to 1 means more similar, ignores vector length.

$$
\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert}
$$


In [ ]:
from fastembed import TextEmbedding
import numpy as np

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [
    ("car repair", "automobile maintenance"),                    # synonyms
    ("cheap flights to New York", "affordable airfare to NYC"),  # paraphrase
    ("cheap flights to New York", "best pizza in Chicago"),      # unrelated
]

for query, document in pairs:
    query_vec = list(model.embed([query]))[0]
    doc_vec = list(model.embed([document]))[0]
    score = cosine_similarity(query_vec, doc_vec)
    print(f"{score:.3f}  |  {query!r}  vs  {document!r}")

# Expected:
#   0.733  car repair vs automobile maintenance
#   0.821  cheap flights to New York vs affordable airfare to NYC
#   0.332  cheap flights to New York vs best pizza in Chicago

Try a polysemy case, same word, more than one meaning:


In [ ]:
polysemy_pairs = [
    ("apple stock", "shares of a tech company"),
    ("apple stock", "a crisp red fruit"),
    ("Apple Inc. stock price", "shares of a tech company"),
]

for query, document in polysemy_pairs:
    query_vec = list(model.embed([query]))[0]
    doc_vec = list(model.embed([document]))[0]
    score = cosine_similarity(query_vec, doc_vec)
    print(f"{score:.3f}  |  {query!r}  vs  {document!r}")

### Distance metrics

| Metric | Common use |
|--------|----------|
| Cosine | Text similarity, NLP models |
| Dot product | Vectors already normalized to unit length |
| Euclidean (L2) | Image embeddings, spatial data |
| Manhattan (L1) | Grid-like or count-based data |

## Similarity alone is not enough

| Pair | Cosine similarity |
|------|-------------------|
| "dog bites man" vs "man bites dog" | 0.907 |
| "safe for kids" vs "harmful to kids" | 0.779 |
| "dog bites man" vs "a canine attacked a person" | 0.570 |

Word order and negation can score high even when the meaning flips. Similarity also fails on exact codes:


In [ ]:
from fastembed import TextEmbedding
import numpy as np

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "SKU-48291 issue"
candidates = ["SKU-48292", "SKU-48291", "SKU-48290"]  # wrong, correct, wrong

query_vec = list(model.embed([query]))[0]

for candidate in candidates:
    candidate_vec = list(model.embed([candidate]))[0]
    score = cosine_similarity(query_vec, candidate_vec)
    print(f"{score:.3f}  |  {candidate}")

# Real output:
#   0.730  SKU-48292  (wrong product)
#   0.734  SKU-48291  (correct product)
#   0.765  SKU-48290  (wrong product, scores HIGHEST)

The fix is a payload filter, not similarity: restrict to points where `sku` equals `"SKU-48291"`. Module 2 covers filters. Module 3 covers hybrid search (dense + sparse vectors).

## Further reading

- [Distance Metrics](https://qdrant.tech/course/essentials/day-1/distance-metrics/)
- [Vector Embeddings Explained](https://qdrant.tech/articles/what-are-embeddings/)
- [FastEmbed](https://qdrant.tech/articles/fastembed/)

Next: `Module2.ipynb`, first Qdrant collection: points, payloads, and your first query.
